# 05 — Demanding case: breast cancer & the curse

*Module 01 · k-Nearest Neighbours — notebook 5 of 6*

The first four notebooks lived on a 2-D toy so every step was visible. This one is the test: a real
dataset, the full workflow, and an honest accounting of where k-NN wins and where it breaks. We
classify breast tumours as malignant or benign from 30 cell-nucleus measurements — then we
deliberately break the model to feel the **curse of dimensionality**.

This is the demanding case, so nothing is hidden: we scale without leaking, choose k by
cross-validation, evaluate once, and read the error that actually matters in a cancer screen.

**Prerequisites:** notebooks 01–04 (the k-NN chapter), and from module 00: notebook 06 (the majority
baseline), notebook 07 (confusion matrix, precision & recall), notebook 08 (the decision threshold),
notebook 10 (cross-validation), notebook 11 (Pipelines & standardization).

**What you will be able to do**

- Run the full honest workflow on a real 30-D dataset.
- Standardize inside a `Pipeline` so no test information leaks into training.
- Choose k by cross-validation, and evaluate the choice exactly once on a sealed test set.
- Read the confusion matrix for the error that matters — a missed malignancy.
- Feel the curse of dimensionality, and explain why it sinks k-NN.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    pairwise_distances,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course import viz
from ml_course.colors import COLORS

# Fix the seed so every run tells the same story.
np.random.seed(0)
viz.use_course_style()

data = load_breast_cancer()
X, y = data.data, data.target  # 0 = malignant, 1 = benign
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=0
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

counts = {str(name): int(n) for name, n in zip(data.target_names, np.bincount(y), strict=True)}
print(f"shape: {X.shape}   train/test: {X_train.shape[0]}/{X_test.shape[0]}")
print(f"classes: {counts}")

## The dataset: a real screening task

The Wisconsin breast cancer dataset holds **569 tumours**, each described by **30 numeric features** —
measurements of cell nuclei from a digitized image (radius, texture, area, smoothness, and so on).
Each tumour is labelled **malignant** or **benign**. This is a screening problem, and screening
problems carry a sharp asymmetry: calling a malignant tumour benign — a missed cancer — is far more
costly than a false alarm. Hold on to that; accuracy alone will not capture it.

As always, we look before we model (the habit from notebook 03).

In [ ]:
df = pd.DataFrame(X, columns=data.feature_names)
print("class balance:", counts)
print("\nranges of a few features (note how different their scales are):")
df[["mean radius", "mean area", "mean smoothness"]].agg(["min", "max"]).round(2)

**Read the output.** Two things to carry forward. First, the classes are mildly imbalanced — 357
benign to 212 malignant — so a "predict benign always" baseline would already score about 63 %;
accuracy must be judged against that (notebook 06). Second, the feature ranges span orders of
magnitude: a mean *area* runs into the hundreds while a mean *smoothness* sits near 0.1. Because k-NN
is pure distance (notebook 2), the large-range features would dominate entirely unless we standardize
— and we must do it **inside** cross-validation so no fold sees the others' statistics (notebook 11).

## The workflow

We assemble the standard honest pipeline:

- **`Pipeline(StandardScaler, KNeighborsClassifier)`** — bundling the scaler with the classifier so
  that, inside cross-validation, the scaler is re-fit on each training fold alone. No validation or
  test statistics leak into the scaling (notebook 11).
- **Cross-validation on the training set** to choose k (notebook 10).
- **One** evaluation on the sealed test set, at the very end.

First, does standardization actually earn its place on this dataset?

In [ ]:
raw_cv = cross_val_score(
    KNeighborsClassifier(n_neighbors=7), X_train, y_train, cv=cv
).mean()
scaled_cv = cross_val_score(
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)), X_train, y_train, cv=cv
).mean()

print(f"raw (unscaled) k-NN CV accuracy:         {raw_cv:.3f}")
print(f"Pipeline(StandardScaler, k-NN) CV acc.:  {scaled_cv:.3f}")

**Read the output.** Standardizing lifts cross-validation accuracy from 0.935 to 0.970. The scale
trap from notebook 2 was not a toy concern — on real data with thirty differently-scaled features,
leaving them unscaled hands the vote to whichever features happen to carry big numbers. The Pipeline
fixes it, and fixes it without leaking.

In [ ]:
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31]
cv_scores = [
    cross_val_score(
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
        X_train,
        y_train,
        cv=cv,
    ).mean()
    for k in ks
]
best_k = ks[int(np.argmax(cv_scores))]
print(f"CV-chosen k = {best_k}   (CV accuracy {max(cv_scores):.3f})")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(ks, cv_scores, color=COLORS["model"], marker="o", linewidth=2)
ax.axvline(best_k, color=COLORS["highlight"], linestyle="--", label=f"CV-best k = {best_k}")
ax.set_xlabel("k (neighbours)")
ax.set_ylabel("cross-validation accuracy")
ax.legend()
plt.show()

**Read the figure.** Cross-validation accuracy (training data only) peaks at **k = 7**, on a gentle
plateau — anything from about 5 to 15 is nearly as good. We take k = 7. This choice used only the
training set; the test set is still sealed.

In [ ]:
final_model = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)
).fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f"test accuracy:     {final_model.score(X_test, y_test):.3f}")
print(f"malignant recall:  {recall_score(y_test, y_pred, pos_label=0):.3f}")
print(f"benign: precision {precision_score(y_test, y_pred):.3f}  "
      f"recall {recall_score(y_test, y_pred):.3f}  F1 {f1_score(y_test, y_pred):.3f}")

cm = confusion_matrix(y_test, y_pred)
viz.plot_confusion_matrix(cm, class_names=list(data.target_names))
plt.show()

**Read the figure.** Test accuracy is 0.947 — strong at first glance. But read the confusion matrix
for the error that matters. Of 64 malignant tumours, the model labels **7 as benign**: a recall on
the malignant class of only **0.891**. Those seven are missed cancers — exactly the error a screen
must avoid. Accuracy hid them, because the benign class (which the model handles almost perfectly)
dominates the average. In practice you would lower the decision threshold to catch more malignancies
at the cost of more false alarms (notebook 08); the right trade-off depends on what a miss costs
versus a false positive.

## When k-NN is the right tool — and when it is not

On this problem k-NN does well: a few hundred samples, thirty standardized features, and a distance
that genuinely reflects similarity. That is k-NN's sweet spot. Its costs are the ones we met in
notebook 1: prediction is slow on large datasets (every query is compared against every stored point)
and the whole training set lives in memory. And there is one failure mode we have described but not
yet felt — **high dimensionality**. Let us provoke it on purpose.

## The curse of dimensionality

Here is the unsettling fact about high-dimensional space: as you add dimensions, the distance to your
nearest neighbour and the distance to your farthest neighbour drift closer and closer together. Past
enough dimensions, every point is roughly the same distance from every other — and "nearest" stops
carrying information. Since k-NN is nothing but "nearest", this is fatal.

We demonstrate it cleanly: keep the 30 real features and append columns of pure noise, then watch two
things — k-NN's accuracy, and the ratio of the nearest distance to the farthest. (The Pipeline
re-standardizes every column, so the noise enters at the signal's unit scale. We track accuracy by
cross-validation on the **training** set, not the test set: this is an illustration of a geometric
effect, not a second look at our sealed test data.)

In [ ]:
rng = np.random.default_rng(0)
scaler = StandardScaler().fit(X_train)
Xtr_s, Xte_s = scaler.transform(X_train), scaler.transform(X_test)

noise_dims = [0, 30, 100, 300, 1000, 2000]
curse_acc, near_far = [], []
for d in noise_dims:
    if d:
        Xtr_noisy = np.hstack([X_train, rng.standard_normal((X_train.shape[0], d))])
        a_tr = np.hstack([Xtr_s, rng.standard_normal((Xtr_s.shape[0], d))])
        a_te = np.hstack([Xte_s, rng.standard_normal((Xte_s.shape[0], d))])
    else:
        Xtr_noisy, a_tr, a_te = X_train, Xtr_s, Xte_s
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
    curse_acc.append(cross_val_score(pipe, Xtr_noisy, y_train, cv=cv).mean())
    dmat = pairwise_distances(a_te, a_tr)
    near_far.append(float((dmat.min(axis=1) / dmat.max(axis=1)).mean()))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(noise_dims, curse_acc, color=COLORS["model"], marker="o", linewidth=2)
axes[0].set_xlabel("pure-noise features added")
axes[0].set_ylabel("cross-validation accuracy")
axes[0].set_title("k-NN accuracy collapses")
axes[1].plot(noise_dims, near_far, color=COLORS["error"], marker="s", linewidth=2)
axes[1].set_xlabel("pure-noise features added")
axes[1].set_ylabel("mean nearest / farthest distance")
axes[1].set_title("distances concentrate (ratio -> 1)")
fig.tight_layout()
plt.show()

**Read the figure.** Left: as we bury the 30 real features under more and more noise, cross-validation
accuracy slides from 0.970 down toward 0.77 — the signal is still there, but k-NN can no longer find
it. Right: the reason. The mean ratio of nearest-distance to farthest-distance climbs from about 0.12
toward 0.91 — with 2000 noise features the closest training point is nearly as far as the farthest.
As that ratio approaches 1, the "nearest" neighbours become an increasingly arbitrary selection and
the vote carries less and less signal (distance concentration; Beyer et al., 1999). We changed nothing about
the tumours — only added noise, and that was enough to dissolve the notion of "near". Real datasets
are rarely pure noise, but they routinely carry many weak or irrelevant features, which apply the
same pressure.

## Your turn

1. The Pipeline standardizes inside each cross-validation fold rather than once on the whole dataset
   beforehand. Why does standardizing *then* splitting leak information, and which way would it bias
   your CV score?
2. The final model misses 7 of 64 malignant tumours. What concrete change would catch more of them,
   and what would it cost (notebook 08)?
3. In your own words: why does the nearest/farthest distance ratio approaching 1 make k-NN useless?
   What does "nearest" even mean when every point is equally far?

## What you built

You ran k-NN's first real engagement, end to end and honestly: you looked at the data, standardized
it inside a Pipeline so nothing leaked, chose k by cross-validation, and spent your one look at the
test set — then read past the 0.947 accuracy to the number that matters, the 7 missed malignancies.
Finally you felt the curse of dimensionality: burying the signal in noise sent accuracy down and the
nearest/farthest ratio up toward 1, and you can now say *why* high dimensions break a distance-based
method.

**Vocabulary**

- **the honest workflow** — look → Pipeline (scale without leaking) → choose k by CV → evaluate once →
  analyse errors → state limits.
- **recall on the costly class** — here, the fraction of malignant tumours correctly caught; the
  number a screen lives or dies by.
- **the curse of dimensionality** — in high dimensions distances concentrate and "nearest" loses
  meaning.
- **distance concentration** — nearest and farthest distances converge as dimensions grow (the
  near/far ratio → 1).
- **when to use k-NN** — moderate n, low-to-moderate dimensionality, standardized, a meaningful
  distance; not large n (slow predict) or high-dimensional/noisy data (the curse).

## References

- W. N. Street, W. H. Wolberg, O. L. Mangasarian (1993). *Nuclear feature extraction for breast tumor
  diagnosis.* IS&T/SPIE — the Breast Cancer Wisconsin (Diagnostic) dataset, distributed via the UCI ML
  Repository and `sklearn.datasets.load_breast_cancer`.
- K. Beyer, J. Goldstein, R. Ramakrishnan, U. Shaft (1999). *When is "nearest neighbor" meaningful?*
  ICDT — the distance-concentration result behind the curse.
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*,
  §2.2.3. DOI: [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).
- T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning*, §2.5 & §13.3.
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

---

*Previous: 04 — The estimator & its parameters* · *Next: 06 — Advanced: distances & choosing k*